<h1 align='center'>Prediccion de Precios de Propiedades Inmobiliarias</h1>
<h2 align='center'>Mercado Australiano — Regresion Avanzada con PyCaret</h2>

# Tabla de Contenido

1. [Introduccion](#Introduccion)
2. [Configuracion e Instalacion](#Configuracion-e-Instalacion)
3. [Entendimiento de los Datos](#Entendimiento-de-los-Datos)
4. [Manipulacion y Limpieza de Datos](#Manipulacion-y-Limpieza-de-Datos)
    1. [Dropping Data](#Dropping-Data)
    2. [Imputacion de Nulos](#Imputacion-de-Nulos)
    3. [Tratamiento de Outliers](#Tratamiento-de-Outliers)
    4. [Derived Data](#Derived-Data)
5. [Analisis de Datos](#Analisis-de-Datos)
    1. [Analisis Univariable](#Analisis-Univariable)
        1. [Plot Numeric Data](#Plot-Numeric-Data)
        2. [Plot Categorical Data](#Plot-Categorical-Data)
    2. [Analisis Bivariable](#Analisis-Bivariable)
    3. [Analisis Multivariable](#Analisis-Multivariable)
6. [Preparacion de Datos y Modelado](#Preparacion-de-Datos-y-Modelado)
    1. [Encoding de Variables](#Encoding-de-Variables)
    2. [Splitting data into Train Test](#Splitting-data-into-Train-Test)
    3. [Feature Scaling — StandardScaler](#Feature-Scaling)
    4. [Feature Engineering — RFE y VIF](#Feature-Engineering)
7. [Construccion del Modelo](#Construccion-del-Modelo)
    1. [Ridge Regression](#Ridge-Regression)
    2. [Lasso Regression](#Lasso-Regression)
    3. [ElasticNet Regression](#ElasticNet-Regression)
    4. [Comparacion de Modelos](#Comparacion-de-Modelos)
    5. [Analisis de Residuos](#Analisis-de-Residuos)
    6. [Importancia de Variables](#Importancia-de-Variables)
    7. [Prediccion en Test Set](#Prediccion-en-Test-Set)
8. [Conclusiones y Observaciones](#Conclusiones-y-Observaciones)

# Introduccion

### Integrantes de Grupo:
* Rios Nuñez David Samuel
* Rivera Quisberth Juan Enrique
* Terceros Beltran Oscar Alvaro
* Torrez Azuga Marcelo

---

## Entendimiento del Negocio

Una empresa de analisis inmobiliario busca ingresar al mercado australiano de bienes raices. La empresa utiliza modelos de datos para identificar propiedades cuyo precio de mercado esta por debajo de su valor real, con el objetivo de adquirirlas y generar retornos positivos. Este modulo de valuacion forma parte de una plataforma de gestion de alquileres que requiere estimaciones de precio confiables para alimentar los modulos de segmentacion de mercado.

Se cuenta con un dataset de ventas de casas en Australia con 81 variables que describen caracteristicas fisicas, de calidad y de ubicacion de cada propiedad. Se debe construir un modelo de regresion regularizada para predecir el valor real de las propiedades.

La gerencia necesita responder:

* **¿Que variables son mas significativas para predecir el precio de venta?**
* **¿Que tan bien describen esas variables el precio?**
* **¿Cual es el valor optimo de lambda (alpha) para Ridge y Lasso?**

### Objetivo del Negocio

Modelar el precio de venta (`SalePrice`) usando variables independientes disponibles. El modelo permitira a la gerencia entender como varian los precios con las caracteristicas de las propiedades, manipular la estrategia de inversion y comprender la dinamica de precios del mercado australiano.

# Configuracion e Instalacion

In [1]:
# Importar Librerias Requeridas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pylab
import seaborn as sns
import sys
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

# Statsmodels
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson

# PyCaret - Regresion
from pycaret.regression import (setup as pycaret_setup, create_model, tune_model,
                                  predict_model, plot_model, pull, compare_models)

pd.options.display.float_format = '{:.2f}'.format
print('Librerias cargadas correctamente.')

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Cargar dataset desde archivo CSV
DATA_FILE_PATH = '_data/dataset.csv'
raw_data = pd.read_csv(DATA_FILE_PATH)
print(f'Dataset cargado: {raw_data.shape[0]} filas, {raw_data.shape[1]} columnas')
raw_data.head()

# Entendimiento de los Datos

In [ ]:
print('=== Dimensiones ===')
print(f'Filas: {raw_data.shape[0]} | Columnas: {raw_data.shape[1]}')
print('\n=== Tipos de Datos ===')
print(raw_data.dtypes.value_counts())
print('\n=== Estadisticas Descriptivas (primeras 5 columnas numericas) ===')
raw_data.describe()

In [ ]:
# Porcentaje de nulos por columna
null_pct = (raw_data.isnull().sum() / len(raw_data) * 100).sort_values(ascending=False)
null_pct_nonzero = null_pct[null_pct > 0]
print(f'Columnas con valores nulos: {len(null_pct_nonzero)}\n')
print(null_pct_nonzero.to_string())

# Heatmap de nulos
plt.figure(figsize=(16, 5))
sns.heatmap(raw_data[null_pct_nonzero.index].isnull(),
            yticklabels=False, cbar=False, cmap='viridis')
plt.title('Mapa de Valores Nulos por Columna', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Distribucion de la variable objetivo SalePrice
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(raw_data['SalePrice'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribucion de SalePrice')
axes[0].set_xlabel('SalePrice (USD)')
axes[0].set_ylabel('Frecuencia')

axes[1].hist(np.log1p(raw_data['SalePrice']), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Distribucion de log(SalePrice + 1)')
axes[1].set_xlabel('log(SalePrice)')
axes[1].set_ylabel('Frecuencia')

plt.suptitle('SalePrice: Original vs Transformacion Logaritmica', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Skewness SalePrice:        {raw_data["SalePrice"].skew():.3f}')
print(f'Skewness log(SalePrice):   {np.log1p(raw_data["SalePrice"]).skew():.3f}')
print('La distribucion log es mas simetrica -> mejor para modelos lineales')

In [ ]:
# Identificar tipos de columnas
num_cols = raw_data.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = raw_data.select_dtypes(include=['object']).columns.tolist()
print(f'Variables numericas:   {len(num_cols)}')
print(f'Variables categoricas: {len(cat_cols)}')
print(f'\nCategoricas: {cat_cols}')

# Manipulacion y Limpieza de Datos

El proceso de limpieza sigue estos pasos:
1. Eliminacion de columnas con alta proporcion de nulos y columnas no predictivas
2. Imputacion de valores nulos con logica semantica del dominio
3. Deteccion y remocion de outliers extremos
4. Creacion de variables derivadas (feature engineering)

## Dropping Data

In [ ]:
df = raw_data.copy()

# Columnas con mas del 40% de nulos - insuficiente informacion para imputar
null_pct_all = df.isnull().sum() / len(df) * 100
high_null_cols = null_pct_all[null_pct_all > 40].index.tolist()
print(f'Columnas eliminadas por >40% nulos:')
for c in high_null_cols:
    print(f'  {c}: {null_pct_all[c]:.1f}%')

df.drop(columns=high_null_cols, inplace=True)

# Id: identificador unico, no tiene poder predictivo
df.drop(columns=['Id'], inplace=True, errors='ignore')

# Verificar y eliminar filas duplicadas
dups = df.duplicated().sum()
print(f'\nFilas duplicadas encontradas: {dups}')
df.drop_duplicates(inplace=True)

print(f'\nShape tras limpieza inicial: {df.shape}')

## Imputacion de Nulos

In [ ]:
# NA semantico en variables categoricas = caracteristica inexistente en la propiedad
cat_none_cols = ['BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'MasVnrType']
for col in cat_none_cols:
    if col in df.columns:
        df[col].fillna('None', inplace=True)

# Electrical: 1 solo nulo, imputar con moda
if 'Electrical' in df.columns:
    df['Electrical'].fillna(df['Electrical'].mode()[0], inplace=True)

# NA semantico en numericas = 0 (sin sotano, sin garage, etc.)
num_zero_cols = ['MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
                 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath',
                 'GarageYrBlt', 'GarageArea', 'GarageCars']
for col in num_zero_cols:
    if col in df.columns:
        df[col].fillna(0, inplace=True)

# LotFrontage: propiedades en el mismo vecindario tienen frentes similares
if 'LotFrontage' in df.columns:
    df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(
        lambda x: x.fillna(x.median()))

# Categoricas restantes: imputar con moda
for col in df.select_dtypes(include='object').columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

# Numericas restantes: imputar con mediana
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

print(f'Nulos restantes: {df.isnull().sum().sum()}')
print(f'Shape actual: {df.shape}')

## Tratamiento de Outliers

In [ ]:
shape_before = df.shape[0]

# Outliers conocidos del dataset Ames: alta area habitable con precio anomalamente bajo
# Representan ventas especiales no representativas del mercado general
outlier_mask = ~((df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000))
df = df[outlier_mask].reset_index(drop=True)

removed = shape_before - df.shape[0]
print(f'Outliers eliminados (GrLivArea > 4000 y SalePrice < 300k): {removed}')

# Visualizar distribucion de SalePrice antes y despues
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].scatter(raw_data['GrLivArea'], raw_data['SalePrice'], alpha=0.4, s=10, color='steelblue')
axes[0].set_title('GrLivArea vs SalePrice (original)')
axes[0].set_xlabel('GrLivArea')
axes[0].set_ylabel('SalePrice')

axes[1].scatter(df['GrLivArea'], df['SalePrice'], alpha=0.4, s=10, color='coral')
axes[1].set_title('GrLivArea vs SalePrice (sin outliers)')
axes[1].set_xlabel('GrLivArea')
axes[1].set_ylabel('SalePrice')

plt.tight_layout()
plt.show()

print(f'Shape final: {df.shape}')

## Derived Data

In [ ]:
# Area total: sotano + primer piso + segundo piso
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

# Total de banos (banos completos = 1, medios banos = 0.5)
df['TotalBath'] = (df['FullBath'] + 0.5 * df['HalfBath'] +
                   df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath'])

# Edad de la casa al momento de la venta
df['HouseAge'] = df['YrSold'] - df['YearBuilt']

# Anos desde la ultima remodelacion
df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']

# Indicadores binarios de presencia de amenidades
df['HasGarage']    = (df['GarageArea'] > 0).astype(int)
df['HasPool']      = (df['PoolArea']   > 0).astype(int)
df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)

derived_cols = ['TotalSF', 'TotalBath', 'HouseAge', 'RemodAge',
                'HasGarage', 'HasPool', 'HasFireplace']
print('Variables derivadas creadas:')
df[derived_cols].describe()

# Analisis de Datos

## Analisis Univariable

### Plot Numeric Data

In [ ]:
num_to_plot = ['SalePrice', 'GrLivArea', 'TotalSF', 'LotArea', 'TotalBath',
               'OverallQual', 'YearBuilt', 'HouseAge', 'GarageArea', 'TotalBsmtSF',
               '1stFlrSF', 'MasVnrArea', 'WoodDeckSF', 'OpenPorchSF', 'LotFrontage']

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()
for i, col in enumerate(num_to_plot):
    if col in df.columns:
        axes[i].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.8)
        axes[i].set_title(col, fontsize=9)

plt.suptitle('Distribucion de Variables Numericas', fontsize=14)
plt.tight_layout()
plt.show()

### Plot Categorical Data

In [ ]:
cat_to_plot = ['MSZoning', 'Neighborhood', 'BldgType', 'HouseStyle',
               'SaleType', 'SaleCondition', 'ExterQual', 'KitchenQual']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()
for i, col in enumerate(cat_to_plot):
    if col in df.columns:
        vc = df[col].value_counts().head(10)
        axes[i].bar(vc.index, vc.values, color='coral', edgecolor='white')
        axes[i].set_title(col, fontsize=10)
        axes[i].tick_params(axis='x', rotation=45, labelsize=8)

plt.suptitle('Frecuencia de Variables Categoricas', fontsize=14)
plt.tight_layout()
plt.show()

## Analisis Bivariable

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Scatter: SalePrice vs variables numericas clave
scatter_data = [('GrLivArea', 'steelblue'), ('TotalSF', 'coral'), ('YearBuilt', 'green')]
for ax, (col, color) in zip(axes[0], scatter_data):
    ax.scatter(df[col], df['SalePrice'], alpha=0.4, color=color, s=12)
    ax.set_xlabel(col)
    ax.set_ylabel('SalePrice')
    ax.set_title(f'SalePrice vs {col}')

# Boxplot: SalePrice por variables categoricas (usando seaborn para mayor compatibilidad)
sns.boxplot(data=df, x='OverallQual', y='SalePrice', ax=axes[1, 0], color='steelblue')
axes[1, 0].set_title('SalePrice por OverallQual')
axes[1, 0].set_xlabel('OverallQual')
axes[1, 0].set_ylabel('SalePrice')

sns.boxplot(data=df, x='BldgType', y='SalePrice', ax=axes[1, 1], color='coral')
axes[1, 1].tick_params(axis='x', rotation=30, labelsize=8)
axes[1, 1].set_title('SalePrice por BldgType')
axes[1, 1].set_xlabel('BldgType')

# Top 15 correlaciones con SalePrice
num_df = df.select_dtypes(include=[np.number])
corr_sp = num_df.corr()['SalePrice'].drop('SalePrice').abs().sort_values(ascending=False).head(15)
corr_sp.sort_values().plot(kind='barh', ax=axes[1, 2], color='steelblue')
axes[1, 2].set_title('Top 15 Correlacion con SalePrice')

plt.suptitle('Analisis Bivariable', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Analisis Multivariable

In [ ]:
# Heatmap de correlacion entre top variables
num_df = df.select_dtypes(include=[np.number])
top_cols = num_df.corr()['SalePrice'].abs().sort_values(ascending=False).head(16).index
corr_matrix = num_df[top_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Mapa de Correlacion — Top 15 Variables vs SalePrice', fontsize=13)
plt.tight_layout()
plt.show()

# Preparacion de Datos y Modelado

## Encoding de Variables

In [ ]:
df_model = df.copy()

# Encoding ordinal: variables de calidad tienen orden natural (Ex > Gd > TA > Fa > Po > None)
quality_map    = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
finish_map     = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0}
exposure_map   = {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0}
gar_finish_map = {'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0}

ordinal_mappings = {
    'ExterQual': quality_map,   'ExterCond': quality_map,
    'BsmtQual':  quality_map,   'BsmtCond':  quality_map,
    'HeatingQC': quality_map,   'KitchenQual': quality_map,
    'GarageQual': quality_map,  'GarageCond': quality_map,
    'BsmtExposure':  exposure_map,
    'BsmtFinType1':  finish_map,  'BsmtFinType2': finish_map,
    'GarageFinish':  gar_finish_map,
}

for col, mapping in ordinal_mappings.items():
    if col in df_model.columns:
        df_model[col] = df_model[col].map(mapping).fillna(0).astype(int)

print(f'Encoding ordinal aplicado a {len(ordinal_mappings)} columnas de calidad')

In [ ]:
# One-hot encoding para variables nominales restantes
# dtype=int asegura compatibilidad con StandardScaler y statsmodels VIF
cat_remaining = df_model.select_dtypes(include='object').columns.tolist()
print(f'Variables para one-hot encoding ({len(cat_remaining)}): {cat_remaining}')

df_model = pd.get_dummies(df_model, columns=cat_remaining, drop_first=True, dtype=int)

print(f'\nShape tras encoding: {df_model.shape}')
print(f'Total de features (sin SalePrice): {df_model.shape[1] - 1}')

## Splitting data into Train Test

In [ ]:
X = df_model.drop(columns=['SalePrice'])
y = df_model['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras')
print(f'Features totales: {X_train.shape[1]}')

## Feature Scaling — StandardScaler

In [ ]:
# StandardScaler normaliza las features para que RFE y VIF sean comparables
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)
print(f'Scaling aplicado. Shape: {X_train_scaled.shape}')

## Feature Engineering — RFE y VIF

In [ ]:
# RFE (Recursive Feature Elimination): seleccionar las N features mas relevantes
N_FEATURES = 20

lr_rfe = LinearRegression()
rfe = RFE(estimator=lr_rfe, n_features_to_select=N_FEATURES)
rfe.fit(X_train_scaled, y_train)

selected_by_rfe = X_train_scaled.columns[rfe.support_].tolist()
print(f'Features seleccionadas por RFE ({len(selected_by_rfe)}):')
for f in selected_by_rfe:
    print(f'  {f}')

X_train_rfe = X_train_scaled[selected_by_rfe]
X_test_rfe  = X_test_scaled[selected_by_rfe]

In [ ]:
# VIF (Variance Inflation Factor): eliminar multicolinealidad
# VIF > 5 indica correlacion problematica entre features

def compute_vif(X):
    vif = pd.DataFrame()
    vif['feature'] = X.columns
    vif['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif.sort_values('VIF', ascending=False).reset_index(drop=True)

X_vif = X_train_rfe.copy()
iteration = 1
while True:
    vif_df = compute_vif(X_vif)
    max_vif = vif_df.iloc[0]['VIF']
    if max_vif <= 5:
        print('VIF OK: todas las features tienen VIF <= 5')
        break
    col_drop = vif_df.iloc[0]['feature']
    print(f'Iteracion {iteration}: eliminando "{col_drop}" (VIF={max_vif:.2f})')
    X_vif = X_vif.drop(columns=[col_drop])
    iteration += 1

print('\nVIF final:')
print(compute_vif(X_vif).to_string(index=False))

In [ ]:
# Crear datasets finales con features seleccionadas
final_features = X_vif.columns.tolist()
print(f'Features finales seleccionadas ({len(final_features)}): {final_features}')

X_train_final = X_train_scaled[final_features]
X_test_final  = X_test_scaled[final_features]

# DataFrames con target para PyCaret
train_final = X_train_final.copy()
train_final['SalePrice'] = y_train.values

test_final = X_test_final.copy()
test_final['SalePrice'] = y_test.values

print(f'\nTrain final: {train_final.shape} | Test final: {test_final.shape}')

# Construccion del Modelo

In [ ]:
# Configurar entorno PyCaret con el set de entrenamiento
# normalize=False porque los datos ya fueron escalados con StandardScaler
exp_reg = pycaret_setup(
    data=train_final,
    target='SalePrice',
    normalize=False,
    session_id=42,
    verbose=False
)
print('PyCaret setup completado.')
print(f'Features de entrenamiento: {len(final_features)}')

## Ridge Regression

In [ ]:
# Ridge penaliza con L2 (suma de cuadrados de coeficientes)
# Mantiene todas las variables pero reduce sus coeficientes
ridge_base = create_model('ridge', verbose=False)

tuned_ridge = tune_model(
    ridge_base,
    optimize='RMSE',
    custom_grid={'alpha': [0.01, 0.1, 1.0, 10.0, 100.0, 500.0]},
    verbose=False
)

# Extraer alpha optimo del pipeline
ridge_params = tuned_ridge.get_params()
ridge_alpha_keys = [k for k in ridge_params if 'alpha' in k and '__' in k]
alpha_ridge = ridge_params[ridge_alpha_keys[0]] if ridge_alpha_keys else ridge_params.get('alpha', 'N/A')
print(f'Alpha optimo Ridge: {alpha_ridge}')

# Metricas en entrenamiento y test
preds_col = 'prediction_label'
train_pr = predict_model(tuned_ridge, data=train_final, verbose=False)
test_pr  = predict_model(tuned_ridge, data=test_final,  verbose=False)

# Compatibilidad PyCaret 2.x / 3.x
if preds_col not in test_pr.columns:
    preds_col = 'Label'

r2_train_ridge = r2_score(train_final['SalePrice'], train_pr[preds_col])
r2_test_ridge  = r2_score(test_final['SalePrice'],  test_pr[preds_col])
rmse_ridge     = np.sqrt(mean_squared_error(test_final['SalePrice'], test_pr[preds_col]))

print(f'R2 Train: {r2_train_ridge:.4f}  |  R2 Test: {r2_test_ridge:.4f}  |  RMSE Test: {rmse_ridge:.2f}')

# Plot residuos Ridge
plot_model(tuned_ridge, plot='residuals', save=False)

## Lasso Regression

In [ ]:
# Lasso penaliza con L1 (suma de valores absolutos de coeficientes)
# Puede reducir coeficientes a exactamente 0 -> seleccion automatica de variables
lasso_base = create_model('lasso', verbose=False)

tuned_lasso = tune_model(
    lasso_base,
    optimize='RMSE',
    custom_grid={'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]},
    verbose=False
)

lasso_params   = tuned_lasso.get_params()
lasso_alpha_keys = [k for k in lasso_params if 'alpha' in k and '__' in k]
alpha_lasso = lasso_params[lasso_alpha_keys[0]] if lasso_alpha_keys else lasso_params.get('alpha', 'N/A')
print(f'Alpha optimo Lasso: {alpha_lasso}')

train_pl = predict_model(tuned_lasso, data=train_final, verbose=False)
test_pl  = predict_model(tuned_lasso, data=test_final,  verbose=False)

preds_col_l = 'prediction_label' if 'prediction_label' in test_pl.columns else 'Label'
r2_train_lasso = r2_score(train_final['SalePrice'], train_pl[preds_col_l])
r2_test_lasso  = r2_score(test_final['SalePrice'],  test_pl[preds_col_l])
rmse_lasso     = np.sqrt(mean_squared_error(test_final['SalePrice'], test_pl[preds_col_l]))

print(f'R2 Train: {r2_train_lasso:.4f}  |  R2 Test: {r2_test_lasso:.4f}  |  RMSE Test: {rmse_lasso:.2f}')

plot_model(tuned_lasso, plot='residuals', save=False)

## ElasticNet Regression

In [ ]:
# ElasticNet combina penalizacion L1 y L2
# l1_ratio controla el balance: 0=Ridge puro, 1=Lasso puro
en_base = create_model('en', verbose=False)

tuned_en = tune_model(
    en_base,
    optimize='RMSE',
    custom_grid={'alpha': [0.001, 0.01, 0.1, 1.0],
                 'l1_ratio': [0.2, 0.5, 0.8]},
    verbose=False
)

en_params = tuned_en.get_params()
en_alpha_keys = [k for k in en_params if 'alpha' in k and '__' in k]
en_l1_keys    = [k for k in en_params if 'l1_ratio' in k]
alpha_en = en_params[en_alpha_keys[0]] if en_alpha_keys else en_params.get('alpha', 'N/A')
l1_en    = en_params[en_l1_keys[0]]   if en_l1_keys    else en_params.get('l1_ratio', 'N/A')
print(f'Alpha optimo ElasticNet: {alpha_en}  |  L1 ratio: {l1_en}')

train_pe = predict_model(tuned_en, data=train_final, verbose=False)
test_pe  = predict_model(tuned_en, data=test_final,  verbose=False)

preds_col_e = 'prediction_label' if 'prediction_label' in test_pe.columns else 'Label'
r2_train_en = r2_score(train_final['SalePrice'], train_pe[preds_col_e])
r2_test_en  = r2_score(test_final['SalePrice'],  test_pe[preds_col_e])
rmse_en     = np.sqrt(mean_squared_error(test_final['SalePrice'], test_pe[preds_col_e]))

print(f'R2 Train: {r2_train_en:.4f}  |  R2 Test: {r2_test_en:.4f}  |  RMSE Test: {rmse_en:.2f}')

plot_model(tuned_en, plot='residuals', save=False)

## Comparacion de Modelos

In [ ]:
results_df = pd.DataFrame({
    'Model':      ['Ridge',        'Lasso',        'ElasticNet'],
    'Alpha':      [alpha_ridge,    alpha_lasso,    alpha_en],
    'R2_Train':   [r2_train_ridge, r2_train_lasso, r2_train_en],
    'R2_Test':    [r2_test_ridge,  r2_test_lasso,  r2_test_en],
    'RMSE_Test':  [rmse_ridge,     rmse_lasso,     rmse_en],
})
results_df['Diferencia_R2'] = (results_df['R2_Train'] - results_df['R2_Test']).round(4)

print('=== Comparacion de Modelos ===')
print(results_df.to_string(index=False))

best_idx = results_df['R2_Test'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
print(f'\nMejor modelo segun R2 Test: {best_model_name}')

# Mapas de modelos y predicciones para uso posterior
model_map = {'Ridge': tuned_ridge, 'Lasso': tuned_lasso, 'ElasticNet': tuned_en}
preds_map = {
    'Ridge':      (train_pr,  test_pr,  preds_col),
    'Lasso':      (train_pl,  test_pl,  preds_col_l),
    'ElasticNet': (train_pe,  test_pe,  preds_col_e),
}
best_model_obj = model_map[best_model_name]
_, best_test_preds, best_pc = preds_map[best_model_name]

## Analisis de Residuos

In [ ]:
# Calcular residuos del mejor modelo en test
y_true = test_final['SalePrice'].values
y_pred = best_test_preds[best_pc].values
residuals = y_true - y_pred

# Durbin-Watson: ~2 = sin autocorrelacion, <1 o >3 = problematico
dw_stat = durbin_watson(residuals)
print(f'Estadistico Durbin-Watson ({best_model_name}): {dw_stat:.4f}')
print('Interpretacion: valores cercanos a 2 indican ausencia de autocorrelacion en residuos')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_pred, residuals, alpha=0.5, color='steelblue', s=15)
axes[0].axhline(y=0, color='red', linestyle='--', lw=1.5)
axes[0].set_xlabel('Valores Predichos')
axes[0].set_ylabel('Residuos')
axes[0].set_title(f'Residuos vs Predichos ({best_model_name})')

axes[1].hist(residuals, bins=40, color='coral', edgecolor='white')
axes[1].set_xlabel('Residuos')
axes[1].set_title('Distribucion de Residuos')

import scipy.stats as scipy_stats
scipy_stats.probplot(residuals, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot de Residuos')

plt.tight_layout()
plt.show()

## Importancia de Variables

In [ ]:
# Extraer coeficientes del estimador final en el pipeline
pipeline_steps = list(best_model_obj.named_steps.values())
final_estimator = pipeline_steps[-1]

if hasattr(final_estimator, 'coef_'):
    coefs = final_estimator.coef_
    # Verificar que la dimension coincide con final_features
    if len(coefs) == len(final_features):
        idx = final_features
    else:
        idx = [f'feature_{i}' for i in range(len(coefs))]

    coef_series = pd.Series(np.abs(coefs), index=idx).sort_values(ascending=False)

    plt.figure(figsize=(10, 7))
    coef_series.head(15).sort_values().plot(kind='barh', color='steelblue')
    plt.title(f'Top 15 Variables por Magnitud de Coeficiente ({best_model_name})')
    plt.xlabel('|Coeficiente| (escalado)')
    plt.tight_layout()
    plt.show()

    print('Las 5 variables mas significativas para predecir SalePrice:')
    for i, (feat, val) in enumerate(coef_series.head(5).items()):
        print(f'  {i+1}. {feat}: {val:.4f}')
else:
    print('El modelo no expone coeficientes directamente.')
    # Alternativa: usar PyCaret nativo
    plot_model(best_model_obj, plot='feature', save=False)

## Prediccion en Test Set

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Real vs Predicho
axes[0].scatter(y_true, y_pred, alpha=0.5, color='steelblue', s=15)
perfect = [y_true.min(), y_true.max()]
axes[0].plot(perfect, perfect, 'r--', lw=2, label='Prediccion perfecta')
axes[0].set_xlabel('SalePrice Real')
axes[0].set_ylabel('SalePrice Predicho')
axes[0].set_title(f'Real vs Predicho — {best_model_name}')
axes[0].legend()

# Distribucion de errores absolutos
mae_vals = np.abs(y_true - y_pred)
axes[1].hist(mae_vals, bins=40, color='coral', edgecolor='white')
axes[1].set_xlabel('Error Absoluto (USD)')
axes[1].set_title('Distribucion de Errores Absolutos')

plt.tight_layout()
plt.show()

print(f'R2 Test:   {r2_score(y_true, y_pred):.4f}')
print(f'RMSE Test: {np.sqrt(mean_squared_error(y_true, y_pred)):.2f}')
print(f'MAE Test:  {np.mean(np.abs(y_true - y_pred)):.2f}')

# Conclusiones y Observaciones

A continuacion se presentan los resultados finales del proyecto y las conclusiones del analisis.

In [ ]:
print('=' * 65)
print('  RESUMEN DE RESULTADOS — REGRESION AVANZADA CON PYCARET')
print('=' * 65)
print(f'Dataset: Ames Housing | Observaciones: {df.shape[0]} | Features finales: {len(final_features)}')
print()
print(f'{"Modelo":<12} {"Alpha":>12} {"R2 Train":>10} {"R2 Test":>10} {"RMSE Test":>12}')
print('-' * 65)
for _, row in results_df.iterrows():
    print(f'{row["Model"]:<12} {str(row["Alpha"]):>12} {row["R2_Train"]:>10.4f} {row["R2_Test"]:>10.4f} {row["RMSE_Test"]:>12.2f}')
print('=' * 65)
print(f'\nMejor modelo: {best_model_name}')
print()

if hasattr(final_estimator, 'coef_'):
    top_vars = pd.Series(np.abs(final_estimator.coef_), index=final_features).sort_values(ascending=False).head(4)
    print('Variables mas significativas para predecir SalePrice:')
    for feat in top_vars.index:
        print(f'  - {feat}')

print()
print('Conclusiones:')
print('  1. Se construyeron 3 modelos de regresion regularizada (Ridge, Lasso, ElasticNet)')
print('  2. RFE redujo de', X_train.shape[1], 'a', N_FEATURES, 'features; VIF elimino las multicolineales')
print(f'  3. {best_model_name} es el mejor modelo con R2 Test = {results_df.loc[best_idx, "R2_Test"]:.4f}')
print('  4. Variables derivadas (TotalSF, TotalBath, HouseAge) mejoraron el poder predictivo')
print('  5. Este modulo provee estimaciones de precio para alimentar la segmentacion de mercado (K-Means)')